# 8 · Tools that work for every schema

**The problem.** Your team keeps writing the same utilities for each record type: an audit log that says what
changed, a data dictionary for the wiki, a debug dump. With interfaces, each utility either knows every type or
leans on reflection that TypeScript erases at runtime.

Here, two things are data you can walk:

1. **Schemas.** A schema is an ordinary value. A tool can read its properties, adjacencies, relations and `unique`
   clauses and generate documentation, forms or code from them.
2. **Instances.** Every instance is *Visitable*. `obj.accept(visitor)` writes the object's properties and entries
   into any visitor you hand it. The object doesn't know what the visitor does with them: a serializer saves them,
   the validator checks them, and your tool can do anything else.

This case study builds both kinds of tool, and they work unchanged on every schema from the earlier case studies.

In [1]:
import { Proxies, Reachable, Schemas } from "@mbse/schemas/Framework";
import type { OfObject } from "@mbse/schemas/Framework/Visitors";

// The address book from case study 2, compactly.
let Contact = new Schemas.OfObject.Builder().properties(
  (p) => p.name("given_name").of((t) => t.as_native(String)),
  (p) => p.name("family_name").of((t) => t.as_native(String)),
  (p) => p.name("vip").of((t) => t.as_native(Boolean)),
).create();
let Address = new Schemas.OfObject.Builder().properties(
  (p) => p.name("street").of((t) => t.as_native(String)),
  (p) => p.name("country_code").of((t) => t.as_native(String)),
).create();
const ContactAddresses = new Schemas.OfRelation.Builder()
  .links("contact", "address")
  .properties((p) => p.name("label").of((t) => t.as_native(String)))
  .unique("address", "label")
  .create();
Contact = new Schemas.OfObject.Builder(Contact).relations((adj) => adj.name("addresses").of(ContactAddresses).me("contact")).update();
Address = new Schemas.OfObject.Builder(Address).relations((adj) => adj.name("residents").of(ContactAddresses).me("address")).update();
Proxies.register("Contact", Contact);
Proxies.register("Address", Address);
Proxies.register("ContactAddresses", ContactAddresses);
const Builders = Proxies.Builders;

## Tool 1: a data dictionary, generated from schemas

`Schemas.OfObject.Data` has `properties` (name to type schema) and `adjacencies` (name to adjacency), both `Map`s. A
relation has `links`, `properties` and `uniques`. `Proxies.name_of(schema)` gives the registered name. That's all a
documentation generator needs:

In [2]:
function typeName(schema: unknown): string {
  return schema instanceof Schemas.OfNative.Data ? (schema.type as { name: string }).name : String((schema as object).constructor.name);
}

function dataDictionary(...schemas: Schemas.OfObject.Data[]): string {
  const lines: string[] = [];
  for (const schema of schemas) {
    lines.push(`### ${Proxies.name_of(schema)}`, "", "| Name | Kind | Details |", "|---|---|---|");
    for (const [name, prop] of schema.properties) lines.push(`| \`${name}\` | property | ${typeName(prop)} |`);
    for (const [name, adjacency] of schema.adjacencies) {
      const relation = adjacency.relation as Schemas.OfRelation.Data;
      const others = relation.links.filter((link) => link !== adjacency.me);
      const keys = relation.uniques.map((u) => `unique(${[...u].sort().join(", ")})`).join("; ") || "no cardinality limit";
      lines.push(`| \`${name}\` | relation \`${Proxies.name_of(relation)}\` as \`${adjacency.me}\` ` +
        `| links ${others.join(", ")}; properties ${[...relation.properties.keys()].join(", ") || "none"}; ${keys} |`);
    }
    lines.push("");
  }
  return lines.join("\n");
}

console.log(dataDictionary(Contact, Address));

### Contact

| Name | Kind | Details |
|---|---|---|
| `given_name` | property | String |
| `family_name` | property | String |
| `vip` | property | Boolean |
| `addresses` | relation `ContactAddresses` as `contact` | links address; properties label; unique(address, label) |

### Address

| Name | Kind | Details |
|---|---|---|
| `street` | property | String |
| `country_code` | property | String |
| `residents` | relation `ContactAddresses` as `address` | links contact; properties label; unique(address, label) |



The same walk could emit a JSON Schema, a Python model or an HTML form. This is the "schemas are values" decision
from case study 1 paying off, and it's how generated bindings for other languages will be produced.

## Tool 2: reading any object, with a visitor

A visitor is a set of small handles. `accept` calls them in a fixed pattern. For each property it calls:

```ts
visitor.property(name, (p) => p.value((v) => v.as_native((n) => n.set(value))));
```

and for each entry of each adjacency:

```ts
visitor.adjacency(name, (a) => a.add((e) => { e.link(other, (k) => k.set(target)); e.property(name, (p) => ...); }));
```

That's the object *writing itself* into the visitor, handle by handle: a property handle `p`, the value handle `v`
which dispatches on kind (`as_native`), the native slot `n`, an adjacency `a`, an entry `e`, and a link `k`. Each
handle passes the next one to a callback instead of returning it. That's why the same object can be written into a
serializer, a validator or a builder without allocating anything in between.

To read any object into plain objects, implement those handles so that `set` stores what it's given:

In [3]:
type Fields = Record<string, unknown>;

/** Handles for one value: `p.value(...)` -> `v.as_native(...)` -> `n.set(value)` store into `target[key]`. */
class Slot {
  constructor(private readonly target: Fields, private readonly key: string) {}

  value(callback: (slot: Slot) => unknown): Slot { // Visitors.OfProperty.value
    callback(this);
    return this;
  }

  as_native(callback: (slot: Slot) => unknown): Slot { // Visitors.OfAny.as_native: native values only here
    callback(this);
    return this;
  }

  set(value: unknown): Slot { // Visitors.OfNative.set, and Visitors.OfLink.set for links
    this.target[this.key] = value;
    return this;
  }
}

/** Handles for one entry: its links and properties land in one row. */
class Entry {
  constructor(private readonly row: Fields) {}

  link(name: string, callback: (slot: Slot) => unknown): Entry {
    callback(new Slot(this.row, name));
    return this;
  }

  property(name: string, callback: (slot: Slot) => unknown): Entry {
    callback(new Slot(this.row, name));
    return this;
  }
}

/** A write-only object visitor: collects what an object writes into it. */
class Record_ {
  readonly properties: Fields = {};
  readonly adjacencies = new Map<string, Fields[]>();

  property(name: string, callback: (slot: Slot) => unknown): Record_ {
    callback(new Slot(this.properties, name));
    return this;
  }

  adjacency(name: string, callback: (adjacency: { add(fill: (entry: Entry) => unknown): unknown }) => unknown): Record_ {
    const rows = this.adjacencies.get(name) ?? [];
    this.adjacencies.set(name, rows);
    callback({
      add(fill) {
        const row: Fields = {};
        fill(new Entry(row));
        rows.push(row);
        return this;
      },
    });
    return this;
  }
}

function record(obj: Proxies.Instance): Record_ {
  const visitor = new Record_();
  obj.accept(visitor as unknown as OfObject); // implements only what accept calls; see below
  return visitor;
}

That's a complete reader in about 60 lines, and it doesn't mention `Contact`, `Address` or any property name.

`accept`'s parameter type is the full `Visitors.OfObject` protocol, which also has methods for *reading* a visitor
(`has`, `properties`, `adjacencies`, ...). This one is only ever *written into*, so it implements just what `accept`
calls, and the cast says so. A reusable visitor, like the builders and serializers, implements the whole protocol
and needs no cast. (The class is `Record_` because `Record` is TypeScript's built-in utility type.)

In [4]:
const home = Builders.Address().street("7 Old Road").country_code("GB").create();
const alice = Builders.Contact()
  .given_name("Alice").family_name("Liddell").vip(false)
  .addresses((x: any) => x.address(home).label("home"))
  .create();

const r = record(alice);
console.log(r.properties);
console.log([...r.adjacencies].map(([name, rows]) => [name, rows.map((row) => ({ ...row, address: (row.address as Proxies.Instance).street }))]));

{ given_name: "Alice", family_name: "Liddell", vip: false }


[
  [ "addresses", [ { address: "7 Old Road", label: "home" } ] ]
]


## Tool 3: an audit log, for any schema

With a generic reader, "what changed?" is a comparison. Linked objects are compared by identity, the same rule the
framework uses for entry equality:

In [5]:
function snapshot(obj: Proxies.Instance): [Fields, Set<string>] {
  const r = record(obj);
  const facts = new Set<string>();
  for (const [name, rows] of r.adjacencies) {
    for (const row of rows) {
      const fields = Object.entries(row)
        .map(([k, v]) => `${k}=${typeof v === "object" && v !== null && "accept" in v ? `#${String((v as Proxies.Instance).identity())}` : String(v)}`)
        .sort();
      facts.add(`${name} {${fields.join(", ")}}`);
    }
  }
  return [r.properties, facts];
}

function changes([propsA, factsA]: [Fields, Set<string>], [propsB, factsB]: [Fields, Set<string>]): void {
  const shown = (v: unknown) => (v === undefined ? "<absent>" : JSON.stringify(v));
  for (const name of [...new Set([...Object.keys(propsA), ...Object.keys(propsB)])].sort()) {
    if (propsA[name] !== propsB[name]) console.log(`  ${name}: ${shown(propsA[name])} -> ${shown(propsB[name])}`);
  }
  for (const fact of [...factsB].filter((f) => !factsA.has(f)).sort()) console.log(`  + ${fact}`);
  for (const fact of [...factsA].filter((f) => !factsB.has(f)).sort()) console.log(`  - ${fact}`);
}

const before = snapshot(alice);
Builders.Contact(alice)
  .family_name("Hargreaves")
  .vip((v: any) => v.clear())
  .addresses((x: any) => x.address((a: any) => a.street("2 New Lane").country_code("GB")).label("home"))
  .update();
changes(before, snapshot(alice));

  family_name: "Liddell" -> "Hargreaves"


  vip: false -> <absent>


  + addresses {address=#3, label=home}


(The `+` entry shows the new address by its in-memory identity. A real audit log would label objects with a symbol,
as snapshots do. Whether the framework should offer such labels to tools is an open question in `FRAMEWORK.md`.)

## Tool 4: dump a whole component

Combine the reader with `Reachable.of` from case study 4 to dump everything connected to an object, whatever its
schema:

In [6]:
for (const obj of Reachable.of(home)) console.log(obj.schema_name().padEnd(8), record(obj as Proxies.Instance).properties);

Address  { street: "7 Old Road", country_code: "GB" }


Contact  { given_name: "Alice", family_name: "Hargreaves" }


Address  { street: "2 New Lane", country_code: "GB" }


## Why visitors, and not `obj.toJSON()`?

- **One protocol, many tools.** Serializers, validators, reachability and your tools all consume the same calls. A
  new implementation of objects (generated classes, say) only needs `accept`, and every tool works with it.
- **Both directions.** The same handles are used to *write* data. A builder is a visitor too: `FromPlain` loads a
  snapshot by making calls like the ones `accept` makes, into a builder.
- **No intermediate structures.** Values flow straight from the object into the tool, and a validator never builds a
  copy of your graph.
- **Schema-agnostic.** Nothing above imports a schema. The tools work on data they've never seen.

## What you learned

- Schemas are data. Walk `properties`, `adjacencies`, `links` and `uniques` to generate documentation or code.
- `obj.accept(visitor)` writes an object into any visitor, through handles passed to callbacks.
- A small write-only visitor reads any object. Diffing, dumping and auditing follow from it.

**Next:** requirements change. Case study 9 evolves schemas (new fields, versions, variants) and marks where the
framework's design is still open.